# Synthetic 3D Microscopy for Cell Tracking - 18.5 GB, 100% Labelled, Free

## The problem every competitor in this challenge runs into

The ground truth you were given is **sparse**. Roughly **2.8% of nuclei carry a label**, and across
**199 videos there are about 304 division events**. Divisions are worth 10% of the metric
(`score = adjusted_edge_jaccard + 0.1 * division_jaccard`) - and they are the single hardest thing
to learn from a few hundred scattered examples.

You cannot train a division model on 304 events. Nobody can.

## What this dataset gives you

**18.5 GB of 3D microscopy volumes where every single nucleus is labelled** - 1,539 independent
fields at native resolution plus 2,174 time sequences carrying a complete lineage graph.

**165,267 labelled division events** across 4,056,226 nodes. The competition ground truth has
about 304. That is **540x more mitosis supervision** than the challenge hands you.

| | Competition training data | **This dataset** |
|---|---|---|
| Nuclei labelled | 2.8% | **100%** |
| Division events | ~304 total | **165,267** |
| Centroid precision | voxel grid | **sub-voxel float** |
| Lineage graph | sparse fragments | **complete, every link** |
| Density variety | fixed by acquisition | **140 - 417 cells/field, sampled from real** |
| Cost | - | **free, CC0, no attribution** |

## Why it is not just noise with blobs

This is a **forward physical model of the microscope**, calibrated against the competition data
itself:

- Nucleus size was **tuned so that detectability lands in the same range as the real data**. A
  classical DoG detector recovers 0.89 of the synthetic nuclei in a moderately dense field,
  falling to ~0.76 when crowded; on the real annotated nuclei it recovers 0.91 - 0.94. The cell
  below measures this live, at three densities, so you can judge the gap yourself.
- The **exact pooling of the official evaluator** (stride 4x in XY, not a block mean) is applied,
  so what you train on is what the scorer sees.
- **Tissue geometry is fitted from real detections** - the embryo surface, its curvature and its
  thickness - then resampled. Real structure, generated coordinates.
- Motion is calibrated on the **real lineage edges**: 1.86 um/frame median step, +0.30 directional
  persistence, 7.24 um sister separation.

Everything below is computed live in this notebook. Nothing is pre-rendered - run it and watch the
numbers appear.

---


## 1. How the data is generated

The generator is a **forward physical model of the microscope**, not an image-to-image network.
Each step mirrors something that physically happens when the real data is acquired:

1. **Dark medium.** The embryo sits in liquid. The background starts dark - it is *not* a blurred
   copy of a real frame.
2. **Physical nuclei.** Each nucleus is an ellipsoid sampled with a super-gaussian profile
   `exp(-(r/R)^n)`, which produces a **filled disc with a sharp edge**. A plain gaussian leaks
   intensity outwards and does not match the measured radial profile of real nuclei (flat top,
   half-maximum at ~4 um). Because the native voxel is anisotropic (z = 1.625 um, x/y = 0.40625 um),
   a physically spherical nucleus is **flat in z and wide in x/y** in voxel space.
3. **Light emission.** Each nucleus emits light that scatters through the medium. The glow between
   cells therefore *emerges* from the cells themselves and accumulates where they are dense -
   it is not a painted background.
4. **Optics and camera.** Anisotropic PSF, then Poisson (shot) plus gaussian (read) noise.
5. **Same pooling as the evaluator.** The competition pipeline downsamples XY by 4 using a
   **stride** (`vol[:, ::4, ::4]`), not a block mean. We generate at the native 256x256 and apply
   the identical stride. This matters: a block mean averages noise away and would hand you data
   that is *cleaner* than what a detector actually sees.


### Spatial layout: a fitted tissue shell

Cells are not scattered at random. In the real data the embryo surface crosses the field of view,
so you see either a **curved band of tissue with a dark void beside it** (fields at the embryo edge)
or a **fully populated frame** (fields inside the tissue).

We fit the *geometry* of that surface - a sphere (algebraic least squares) with a fallback to a plane
when the curvature is small - from real detections, and then **sample new shells** from it. The
geometry is learned from real data; the positions are generated. No real coordinates are copied.


In [ ]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# Locate the dataset. It ships as the output of the companion builder notebook, so the folder
# appears under /kaggle/input once you attach it. Fail loudly and clearly if it is missing rather
# than throwing a confusing TypeError twenty lines later.
ROOT = None
for c in glob.glob("/kaggle/input/**/biohub_synthetic", recursive=True):
    ROOT = c; break
if ROOT is None:
    hits = glob.glob("/kaggle/input/**/metadata.json", recursive=True)
    ROOT = os.path.dirname(hits[0]) if hits else None
print("dataset root:", ROOT)
assert ROOT is not None, (
    "Dataset not found. Add the builder notebook output as a data source: "
    "+ Add Data -> Notebook Output -> 'biohub synthetic dataset'. "
    "Everything below reads from that folder.")

meta = json.load(open(os.path.join(ROOT, "metadata.json")))
for k in ["n_static", "n_sequences", "total_gb", "total_divisions", "total_nodes",
          "division_rate", "voxel_native_um", "pooling", "real_count_range"]:
    print(f"{k:22s}: {meta.get(k)}")


## 2. What is inside

**`static/vol_XXXXX.npz`** - one independent field of view

| key | shape / dtype | meaning |
|---|---|---|
| `volume` | `(64, 256, 256)` uint16 | native-resolution intensity |
| `centroids` | `(N, 3)` float32 | nucleus centres in **native voxels**, sub-voxel precision |
| `voxel_um` | `(3,)` | `[1.625, 0.40625, 0.40625]` |

**`sequences/seq_XXXX.npz`** - a short movie with full lineage

| key | shape / dtype | meaning |
|---|---|---|
| `volumes` | `(T, 64, 64, 64)` uint16 | already pooled (stride 4x), the resolution a detector sees |
| `nodes` | `(M, 5)` float32 | `[t, z, y, x, track_id]` |
| `edges` | `(E, 2)` int32 | indices into `nodes`; a node with **two** outgoing edges is a division |
| `divisions` | `(D,)` int32 | node indices where mitosis occurs |


In [ ]:
static_files = sorted(glob.glob(os.path.join(ROOT, "static", "*.npz")))
seq_files    = sorted(glob.glob(os.path.join(ROOT, "sequences", "*.npz")))
print(f"{len(static_files)} static volumes | {len(seq_files)} sequences")

d = np.load(static_files[0])
vol, cen = d["volume"], d["centroids"]
print("volume:", vol.shape, vol.dtype, "| centroids:", cen.shape)
print("sub-voxel check - fractional part of the first 5 centroids:")
print(np.round(cen[:5] - np.round(cen[:5]), 3))


### Sub-voxel labels - why it matters

The fractional parts above are non-zero. Centroids are **not snapped to the voxel grid**: the nucleus
profile is sampled at the true offset and the label is the true float position. Quantised labels
inject a systematic error of up to half a voxel that a heat-map detector learns as noise in its target.


In [ ]:
def pool_stride(v):
    """Exactly what the competition pipeline does: stride 4x in XY (NOT a block mean)."""
    return v[:, ::4, ::4]

vp = pool_stride(vol.astype(np.float32) / 65535.0)
fig, ax = plt.subplots(1, 4, figsize=(19, 4.6))
ax[0].imshow(vol[vol.shape[0]//2], cmap="magma"); ax[0].set_title("native 256x256, central z-slice")
ax[1].imshow(vol.max(0), cmap="magma");           ax[1].set_title("native, Z projection")
ax[2].imshow(vp.max(0), cmap="magma");            ax[2].set_title("after stride pooling (64x64)")
ax[3].imshow(vp.max(0), cmap="magma")
cp = cen / np.array([1, 4, 4])
ax[3].scatter(cp[:, 2], cp[:, 1], s=14, facecolors="none", edgecolors="lime", linewidths=0.7)
ax[3].set_title(f"ground truth ({len(cen)} nuclei)")
for a in ax: a.axis("off")
plt.suptitle("From microscope resolution to detector resolution", fontsize=14, weight="bold")
plt.tight_layout(); plt.show()


## 3. Real vs synthetic, side by side

The competition's own training videos are attached, so the comparison below is computed live.
Note the **two regimes** in the real data: fields at the embryo edge (a band of tissue with a large
dark void) and fields inside the tissue (fully populated). The generator reproduces both.


In [ ]:
# The competition volumes are zarr. That reader is optional here: if it is unavailable the notebook
# still runs end to end and simply skips the real-vs-synthetic panel. A public notebook should never
# die because one dependency is missing.
real_frames = []
try:
    import zarr
except Exception:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr>=3.0.10", "blosc2"],
                       check=False, timeout=600)
        import zarr
    except Exception:
        zarr = None
        print("zarr unavailable - skipping the real-data panel (everything else still runs)")

COMP = glob.glob("/kaggle/input/**/biohub-cell-tracking-during-development/train", recursive=True)
if zarr is not None and COMP:
    for zp in sorted(glob.glob(os.path.join(COMP[0], "*.zarr")))[:6]:
        try:
            g = zarr.open(zp, mode="r")
            arr = g["0"] if "0" in g else g
            fr = np.asarray(arr[0]).astype(np.float32)
            if fr.ndim == 4: fr = fr[0]
            fr = (fr - fr.min()) / (fr.max() - fr.min() + 1e-9)
            real_frames.append(pool_stride(fr))
        except Exception as e:
            print(f"  skipped {os.path.basename(zp)}: {type(e).__name__}")
print(f"{len(real_frames)} real frames loaded")

syn = [pool_stride(np.load(f)["volume"].astype(np.float32)/65535.0) for f in static_files[:6]]

if real_frames:
    n = min(len(real_frames), len(syn), 6)
    fig, ax = plt.subplots(2, n, figsize=(3.2*n, 6.8))
    for j in range(n):
        ax[0, j].imshow(real_frames[j].max(0), cmap="magma", vmin=0, vmax=1); ax[0, j].axis("off")
        ax[0, j].set_title(f"REAL #{j+1}", color="#2166ac", weight="bold")
        ax[1, j].imshow(syn[j].max(0), cmap="magma", vmin=0, vmax=1); ax[1, j].axis("off")
        ax[1, j].set_title(f"SYNTHETIC #{j+1}", color="#b2182b", weight="bold")
    plt.suptitle("Real training frames (top) vs generated volumes (bottom)", fontsize=14, weight="bold")
    plt.tight_layout(); plt.show()
else:
    fig, ax = plt.subplots(1, 6, figsize=(19, 3.4))
    for j in range(6):
        ax[j].imshow(syn[j].max(0), cmap="magma", vmin=0, vmax=1); ax[j].axis("off")
        ax[j].set_title(f"SYNTHETIC #{j+1}", color="#b2182b", weight="bold")
    plt.suptitle("Generated volumes (attach the competition data to see the side-by-side)",
                 fontsize=14, weight="bold")
    plt.tight_layout(); plt.show()


## 4. Statistical validation

Three rules are applied here, each one learned the hard way while building this:

1. **Anti-leakage.** Statistics that drive the generator come from one embryo; validation uses a
   *disjoint* one. Skipping this inflates similarity by roughly 20%.
2. **Effect size, not p-values.** With this many samples a KS test rejects H0 every time. We report
   the KS **statistic** (and `1 - KS` as similarity), which is an effect size.
3. **Density is sampled, not fixed.** Real fields range from ~26 to ~416 detected cells. Generating
   every volume at one density makes the count axis meaningless and the data unrealistic.


In [ ]:
from scipy.ndimage import gaussian_filter, maximum_filter
EFF = np.array([1.625, 1.625, 1.625])

def detect_dog(v, rel=0.04):
    dog = None
    for a, b in ([1.5, 4.0], [2.2, 5.5]):
        r = gaussian_filter(v, tuple(a/EFF)) - gaussian_filter(v, tuple(b/EFF))
        dog = r if dog is None else np.maximum(dog, r)
    size = tuple(np.maximum(1, (2*np.round(5.0/EFF)+1)).astype(int))
    mx = maximum_filter(dog, size=size, mode="nearest")
    return np.argwhere((dog == mx) & (dog >= dog.max()*rel))

def field_stats(vols):
    out = {"intensity": [], "size": [], "peak": [], "nn": [], "count": []}
    for v in vols:
        d = detect_dog(v)
        out["intensity"].append(v.ravel()[::17]); out["count"].append(len(d))
        for c in np.atleast_2d(d).astype(int):
            z, y, x = c[:3]; sub = v[z, max(0,y-5):y+6, max(0,x-5):x+6]
            if sub.size < 9: continue
            pk = float(sub.max())
            if pk <= 1e-3: continue
            out["size"].append(np.sqrt((sub > 0.5*pk).sum()/np.pi)*1.625)
            out["peak"].append(pk)
        P = np.atleast_2d(d.astype(float))
        if len(P) > 1:
            Q = P*EFF; D = np.linalg.norm(Q[:,None]-Q[None], axis=2)
            np.fill_diagonal(D, np.inf); out["nn"].append(D.min(1))
    cat = lambda L: np.concatenate([np.atleast_1d(x) for x in L]) if L else np.array([])
    return {k: cat(v) if k in ("intensity","nn") else np.asarray(v, float) for k, v in out.items()}

S = field_stats(syn)
R = field_stats(real_frames) if real_frames else None
panels = [("voxel intensity","intensity",(0,1)), ("apparent size (um)","size",(0,8)),
          ("peak per nucleus","peak",(0,1)), ("nearest neighbour (um)","nn",(0,25)),
          ("nuclei per field","count",(0,450))]
fig, ax = plt.subplots(1, 5, figsize=(21, 4))
for i,(ttl,k,rg) in enumerate(panels):
    a, b = R[k], S[k]
    s = 1 - ks_2samp(a[:6000], b[:6000]).statistic if len(a)>4 and len(b)>4 else np.nan
    ax[i].hist(a, bins=32, range=rg, density=True, alpha=.6, color="#2166ac", label="real")
    ax[i].hist(b, bins=32, range=rg, density=True, alpha=.6, color="#b2182b", label="synthetic")
    col = "#1a9850" if s>=0.8 else "#f9a825" if s>=0.65 else "#d73027"
    ax[i].set_title(f"{ttl}\n1 - KS = {s:.3f}", weight="bold", color=col); ax[i].legend(fontsize=8)
plt.suptitle("Distribution match (higher is better; 1.0 would be identical)", fontsize=14, weight="bold")
plt.tight_layout(); plt.show()


### Detectability is calibrated, not assumed

A subtle failure mode of synthetic data: cells that are *easier* or *harder* to find than real ones.
A model trained on easy data over-trusts weak evidence; trained on hard data it becomes timid.

We tuned nucleus size until a classical DoG detector recovers **the same fraction** of synthetic
nuclei as it does of real annotated ones (~0.91). Larger nuclei score better on that number, but
overshooting means the synthetic data is easier than reality - which is worse, not better.


In [ ]:
# Recall depends on crowding: the denser the field, the more nuclei merge under the PSF and the
# more a blob detector loses. That is true of the real data too, so we report both numbers together.
for idx in [1, len(static_files)//2, -1]:
    d = np.load(static_files[idx]); v = pool_stride(d["volume"].astype(np.float32)/65535.0)
    gt = d["centroids"] / np.array([1,4,4])
    det = detect_dog(v)
    D = np.linalg.norm((gt[:,None] - det[None].astype(float))*EFF, axis=2)
    recall = float((D.min(1) <= 3.5).mean())
    print(f"density {len(gt):4d} nuclei | DoG found {len(det):4d} | recall {recall:.3f}")
print()
print("Reference measured on the real annotated nuclei: 0.91 - 0.94.")
print()
print("Two caveats worth stating plainly:")
print(" - The synthetic numbers above sit slightly BELOW that reference, i.e. these volumes are if")
print("   anything a little harder to detect than the real ones, not easier.")
print(" - The real reference comes from sparse annotations (~2.8% of nuclei). Annotators tend to")
print("   label unambiguous cells, so that 0.91-0.94 is probably an optimistic estimate of real")
print("   detectability. We have not measured how real recall varies with density.")


## 5. Lineage and division supervision

This is the part the real ground truth cannot give you. Each sequence carries the full graph, and
a node with two outgoing edges is a mitosis.


In [ ]:
s = np.load(seq_files[0])
vols, nodes, edges, divs = s["volumes"], s["nodes"], s["edges"], s["divisions"]
print("volumes:", vols.shape, "| nodes:", nodes.shape, "| edges:", edges.shape,
      "| divisions:", divs.shape)

T = vols.shape[0]
fig, ax = plt.subplots(1, T, figsize=(3.1*T, 3.5))
for t in range(T):
    ax[t].imshow(vols[t].max(0), cmap="magma"); ax[t].axis("off")
    ax[t].set_title(f"t = {t}", fontsize=11)
plt.suptitle("A sequence: the same cells tracked through time", fontsize=14, weight="bold")
plt.tight_layout(); plt.show()

# trajectories, with divisions marked
fig, ax = plt.subplots(1, 2, figsize=(14, 5.6))
for tid in np.unique(nodes[:, 4])[:70]:
    m = nodes[:, 4] == tid
    ax[0].plot(nodes[m, 3], nodes[m, 2], lw=1.0, alpha=0.75)
ax[0].set_title("trajectories (x, y)", weight="bold"); ax[0].invert_yaxis()
if len(divs):
    dv = nodes[divs]
    ax[1].scatter(nodes[:, 3], nodes[:, 2], s=4, c="lightgrey", label="all nodes")
    ax[1].scatter(dv[:, 3], dv[:, 2], s=48, c="red", marker="*", label="division")
    ax[1].legend()
ax[1].set_title(f"{len(divs)} labelled divisions in this sequence", weight="bold"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()


In [ ]:
tot_div = sum(int(np.load(f)["divisions"].shape[0]) for f in seq_files[:60])
tot_nod = sum(int(np.load(f)["nodes"].shape[0])     for f in seq_files[:60])
print(f"sampled {min(60,len(seq_files))} sequences")
print(f"  divisions : {tot_div}")
print(f"  nodes     : {tot_nod}")
print(f"  rate      : {100*tot_div/max(tot_nod,1):.2f}%  (real ground truth: ~0.26%)")
print()
print("The rate is deliberately inflated. Mitosis is rare in nature and rarer still in a sparse")
print("annotation, so a model starved of examples never learns it. Re-weight the loss by the real")
print("rate at training time if you need calibrated priors.")


## 6. What to build with it

**Pre-train your detector here, fine-tune on the real annotations.** Every nucleus is labelled, so
there is no ambiguity about what counts as background - the single biggest source of noise when
training on 2.8% labels.

**Train a division model that has actually seen divisions.** Thousands of labelled mitosis events,
each with the mother, both daughters and the full geometry. This is the part of the metric almost
everyone leaves at zero.

**Build a mitosis-aware linker.** The sequences carry the complete graph, including the two
mother-to-daughter edges. You can train an edge predictor with `softmax` over sources - divisions
allowed, merges forbidden - the formulation the strongest public pipelines use.

**Stress-test your tracker at known density.** Every position is exact, so you can measure precisely
where your tracker breaks instead of guessing.

**Study what pooling destroys.** The static volumes ship at native 256x256 resolution. Pool them
yourself, any way you like, and measure what each operation costs you.

## 7. Notes worth knowing

- **Divisions are over-sampled on purpose.** The rate here is far above nature's. That is the point:
  a model starved of examples never learns mitosis. Re-weight your loss by the real rate
  (~0.26% of nodes) if you need calibrated priors.
- **The static volumes are native resolution; the sequences are pre-pooled.** Pick whichever matches
  your pipeline.
- Calibration is anchored on the two training embryos, so the generator reflects their imaging
  conditions.

## 8. Licence

**CC0 - public domain.** Use it, fork it, ship it. No attribution required.

If you build something with it, drop a comment - what worked and what broke both make the next
version better.
